In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("new_labelled_data.csv")
df.sample(10)

,Unnamed: 0,Food Name,Label
622,622,"diet coke, caffeine free",Junk
340,340,"centrum advance, multivitamin",Not Junk
74,74,"coffee, prepared from grounds",Not Junk
518,518,"pork chops, loin, visible fat eaten",Not Junk
538,538,"apple, with skin",Not Junk
76,76,dumpling chive pork,Not Junk
278,278,"tofu, not silken, soft",Not Junk
520,520,"bulk, pure whey isolate, pistachio ice cream",Not Junk
151,151,"sweet potato, boiled",Not Junk
306,306,"white rice, steamed",Not Junk


Convert labels to integers (0 for Not Junk, 1 for Junk)

In [3]:
label_map = {"Not Junk":0, "Junk":1}
df["Label"] = df["Label"].map(label_map)

BERT requires input in a specific format (tokenized text), so we’ll use the BertTokenizer to preprocess the food item descriptions.

In [4]:
from transformers import RobertaTokenizer

# Load the BERT tokenizer
tokenizer = RobertaTokenizer.from_pretrained("roberta-large")

# Tokenize the data
def encode_data(texts, labels, tokenizer):
    encodings = tokenizer(texts, truncation=True, padding=True, max_length=64)
    return encodings, labels


c:\Python\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Now, create a custom PyTorch Dataset to return the tokenized data in the format BERT expects.

In [5]:
import torch
from torch.utils.data import Dataset

class FoodDataset(Dataset):
    def __init__(self, texts, labels, tokenizer):
        self.encodings = tokenizer(texts, truncation=True, padding=True, max_length=64)
        self.labels = labels
    def __getitem__(self, index):
        item = {key: torch.tensor(val[index]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[index])
        return item
    def __len__(self):
        return len(self.labels)

# Prepare the dataset
training_texts = df["Food Name"].tolist()
training_label = df["Label"].tolist()

# Create the dataset
training_dataset = FoodDataset(training_texts, training_label, tokenizer)

Check if GPU is available

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


Now, load a pre-trained BERT model with a classification head.

In [7]:
from transformers import RobertaForSequenceClassification

model = RobertaForSequenceClassification.from_pretrained("roberta-large", num_labels=2)

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-large and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Move the model to the GPU

In [8]:
model.to(device)

RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 1024, padding_idx=1)
      (position_embeddings): Embedding(514, 1024, padding_idx=1)
      (token_type_embeddings): Embedding(1, 1024)
      (LayerNorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-23): 24 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSelfAttention(
              (query): Linear(in_features=1024, out_features=1024, bias=True)
              (key): Linear(in_features=1024, out_features=1024, bias=True)
              (value): Linear(in_features=1024, out_features=1024, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=1024, out_features=1024, bias=True)
 

Next, we define the training arguments, such as the number of epochs, batch size, etc.

In [9]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir='./results',          # output directory
    num_train_epochs=3,              # number of training epochs
    per_device_train_batch_size=16,   # batch size for training
    per_device_eval_batch_size=16,    # batch size for evaluation
    warmup_steps=500,                # number of warmup steps for learning rate scheduler
    weight_decay=0.01,               # strength of weight decay
    logging_dir='./logs',            # directory for storing logs
    logging_steps=10,
    fp16=True                        # # Mixed precision for better performance on GPUs
)

Now, set up the Trainer class with the model, training arguments, and dataset.

In [10]:
from transformers import Trainer

trainer = Trainer(
    model=model, 
    args=training_args,
    train_dataset=training_dataset,
    eval_dataset=None  # You can add validation data here if available
)

c:\Python\Lib\site-packages\accelerate\accelerator.py:488: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)


Finally, train the model using the Trainer class.

In [11]:
# Train the model
trainer.train()

  0%|          | 0/150 [00:00<?, ?it/s]

{'loss': 0.8642, 'grad_norm': 38.556278228759766, 'learning_rate': 8.000000000000001e-07, 'epoch': 0.2}
{'loss': 0.8703, 'grad_norm': 10.00970458984375, 'learning_rate': 1.8e-06, 'epoch': 0.4}
{'loss': 0.8189, 'grad_norm': 15.54995059967041, 'learning_rate': 2.8000000000000003e-06, 'epoch': 0.6}
{'loss': 0.6922, 'grad_norm': 14.857647895812988, 'learning_rate': 3.8e-06, 'epoch': 0.8}
{'loss': 0.5395, 'grad_norm': 29.813398361206055, 'learning_rate': 4.800000000000001e-06, 'epoch': 1.0}
{'loss': 0.5918, 'grad_norm': 46.619773864746094, 'learning_rate': 5.600000000000001e-06, 'epoch': 1.2}
{'loss': 0.4503, 'grad_norm': nan, 'learning_rate': 6.5000000000000004e-06, 'epoch': 1.4}
{'loss': 0.4218, 'grad_norm': 5.997162818908691, 'learning_rate': 7.5e-06, 'epoch': 1.6}
{'loss': 0.4531, 'grad_norm': 17.48975372314453, 'learning_rate': 8.500000000000002e-06, 'epoch': 1.8}
{'loss': 0.3466, 'grad_norm': 8.090161323547363, 'learning_rate': 9.4e-06, 'epoch': 2.0}
{'loss': 0.2245, 'grad_norm': 4.36

TrainOutput(global_step=150, training_loss=0.47520933945973715, metrics={'train_runtime': 25.3355, 'train_samples_per_second': 94.729, 'train_steps_per_second': 5.921, 'total_flos': 113579134905600.0, 'train_loss': 0.47520933945973715, 'epoch': 3.0})

After training, you can use the model to classify new food items.

In [12]:
# Function to classify new items
def classify_food_item(item):
    # Move tokenizer output tensors to the GPU (if available)
    encoding = tokenizer(item, truncation=True, padding=True, max_length=64, return_tensors='pt').to(device)
    
    # Ensure that the model and data are on the same device
    outputs = model(**encoding)  # Model is already on GPU
    logits = outputs.logits
    
    predicted_class = torch.argmax(logits, dim=1).item()
    return 'Junk' if predicted_class == 1 else 'Not Junk'


In [13]:
# Example usage
new_item = "chocolate"
classification = classify_food_item(new_item)
print(f"The item '{new_item}' is classified as: {classification}")

The item 'chocolate' is classified as: Junk


In [14]:
import pandas as pd

In [15]:
df = pd.read_csv("unlabelled_training_data.csv")
df.head()

,Food Name,Category
0,"Fage, Total, Greek Strained Yoghurt, 5% Fat",Dairy and Egg Products
1,"Bulk, Pure Whey Isolate, Pistachio Ice Cream",Supplements
2,"Eggs, Cooked",Dairy and Egg Products
3,"Centrum Advance, Multivitamin",Supplements
4,"Seven Seas, Omega 3 Capsules, Max Strength",Supplements


In [16]:
df["generated_label"] = df["Food Name"].apply(classify_food_item)

In [17]:
df.sample(20)

,Food Name,Category,generated_label
361,"White Rice, Steamed",Cereal Grains and Pasta,Not Junk
158,Prosciutto,Pork Products,Not Junk
480,"Figs, Fresh",Fruits and Fruit Juices,Junk
641,"Coffee, Prepared From Grounds",Beverages,Not Junk
275,"Diet Coke, Caffeine Free",Beverages,Junk
362,"Chicken Breast, Skin Removed Before Cooking",Poultry Products,Not Junk
310,Kimchi,Vegetables and Vegetable Products,Not Junk
199,"Nectarine, Fresh",Fruits and Fruit Juices,Not Junk
523,"Pistachio Nuts, Raw",Nut and Seed Products,Junk
90,"Carrots, Cooked From Fresh",Vegetables and Vegetable Products,Not Junk


: 